In [10]:
import sys
sys.path.append('../src')

In [11]:
import os
from tensorflow.keras.datasets import fashion_mnist
from PIL import Image
import random

(x_train, _), _ = fashion_mnist.load_data()

output_dir = "../data/raw/Not A Note"
os.makedirs(output_dir, exist_ok=True)

random.seed(42)
indices = random.sample(range(len(x_train)), 150)

for i, idx in enumerate(indices):
    img = Image.fromarray(x_train[idx]).convert("RGB")  # grayscale -> RGB
    img = img.resize((224, 224))
    img.save(f"{output_dir}/fashion_{i}.jpg")

print(f"Saved 150 images to {output_dir}")

Saved 150 images to ../data/raw/Not A Note


In [12]:
from pathlib import Path

data_dir = Path("../data/raw")
for folder in data_dir.iterdir():
    if folder.is_dir():
        count = len(list(folder.glob("*.*")))
        print(f"{folder.name}: {count} images")

Fake Notes: 650 images
Not A Note: 165 images
Real Notes: 950 images


In [13]:
from preprocessing import split_dataset
split_dataset("../data/raw", "../data/processed")

Fake Notes: 520 train, 130 val
Real Notes: 760 train, 190 val
Not A Note: 132 train, 33 val


In [14]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
)

val_datagen = ImageDataGenerator(rescale=1.0/255)

train_generator = train_datagen.flow_from_directory(
    "../data/processed/train",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",   # changed from "binary"
    shuffle=True,
)

val_generator = val_datagen.flow_from_directory(
    "../data/processed/val",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",   # changed from "binary"
    shuffle=False,
)

print(train_generator.class_indices)

Found 1412 images belonging to 3 classes.
Found 353 images belonging to 3 classes.
{'Fake Notes': 0, 'Not A Note': 1, 'Real Notes': 2}


In [15]:
train_generator = train_datagen.flow_from_directory(
    "../data/processed/train",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True,
)

val_generator = val_datagen.flow_from_directory(
    "../data/processed/val",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False,
)

print(train_generator.class_indices)

Found 1412 images belonging to 3 classes.
Found 353 images belonging to 3 classes.
{'Fake Notes': 0, 'Not A Note': 1, 'Real Notes': 2}


In [16]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(3, activation="softmax")(x)   # changed: 3 neurons + softmax

three_class_model = models.Model(inputs, outputs)
three_class_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,163 (8.93 MB)

 Trainable params: 82,179 (321.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [17]:
# Compile and train the model
three_class_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",   # changed from binary_crossentropy
    metrics=["accuracy"]
)

three_class_history = three_class_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

Epoch 1/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 99s 2s/step - accuracy: 0.6856 - loss: 0.6943 - val_accuracy: 0.8725 - val_loss: 0.2971
Epoch 2/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 87s 2s/step - accuracy: 0.8598 - loss: 0.3253 - val_accuracy: 0.9178 - val_loss: 0.2073
Epoch 3/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 87s 2s/step - accuracy: 0.9143 - loss: 0.2371 - val_accuracy: 0.9207 - val_loss: 0.2129
Epoch 4/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 87s 2s/step - accuracy: 0.9263 - loss: 0.1866 - val_accuracy: 0.9462 - val_loss: 0.1397
Epoch 5/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 87s 2s/step - accuracy: 0.9398 - loss: 0.1699 - val_accuracy: 0.9292 - val_loss: 0.1576
Epoch 6/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 89s 2s/step - accuracy: 0.9483 - loss: 0.1503 - val_accuracy: 0.8980 - val_loss: 0.2374
Epoch 7/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.9504 - loss: 0.1290 - val_accuracy: 0.9433 - val_loss: 0.1365
Epoch 8/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 85s 2s/step - accuracy: 0.9582 - loss: 0.1266 - val_accuracy: 0.9178 - val_loss:

In [18]:
from pathlib import Path

Path("../models").mkdir(exist_ok=True)
three_class_model.save("../models/mobilenetv2_3class_currency.keras")
print("3-class model saved successfully!")

3-class model saved successfully!


In [19]:
import numpy as np
from PIL import Image

# Grab one sample from the Not A Note validation folder
test_path = list(Path("../data/processed/val/Not A Note").glob("*.*"))[0]
test_img = Image.open(test_path).convert("RGB").resize((224, 224))
test_array = np.expand_dims(np.array(test_img) / 255.0, axis=0)

pred = three_class_model.predict(test_array)[0]
class_names = ["Fake Notes", "Not A Note", "Real Notes"]  # matches class_indices order

for name, prob in zip(class_names, pred):
    print(f"{name}: {prob*100:.1f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Fake Notes: 1.0%
Not A Note: 59.7%
Real Notes: 39.2%


In [23]:
# Test with a completely different image - simulate testing with a random
# non-training image if you have one, or use another Not A Note validation sample
import random

test_samples = list(Path("../data/processed/val/Not A Note").glob("*.*"))
random.shuffle(test_samples)

for test_path in test_samples[:15]:
    test_img = Image.open(test_path).convert("RGB").resize((224, 224))
    test_array = np.expand_dims(np.array(test_img) / 255.0, axis=0)
    pred = three_class_model.predict(test_array, verbose=0)[0]
    predicted_class = class_names[np.argmax(pred)]
    confidence = np.max(pred)
    print(f"{test_path.name}: predicted={predicted_class} ({confidence*100:.1f}%)")

fashion_119.jpg: predicted=Not A Note (100.0%)
fashion_101.jpg: predicted=Not A Note (100.0%)
fashion_2.jpg: predicted=Not A Note (100.0%)
fashion_148.jpg: predicted=Not A Note (100.0%)
fashion_116.jpg: predicted=Not A Note (99.9%)
fashion_115.jpg: predicted=Not A Note (100.0%)
fashion_1.jpg: predicted=Not A Note (100.0%)
fashion_120.jpg: predicted=Not A Note (100.0%)
fashion_21.jpg: predicted=Not A Note (100.0%)
fashion_88.jpg: predicted=Not A Note (100.0%)
fashion_50.jpg: predicted=Not A Note (100.0%)
fashion_25.jpg: predicted=Not A Note (100.0%)
fashion_3.jpg: predicted=Not A Note (100.0%)
fashion_14.jpg: predicted=Not A Note (100.0%)
fashion_46.jpg: predicted=Not A Note (100.0%)


In [27]:
# Test with any fabric/texture-like image you have saved locally.
# Update this path to point to the actual floral image you tested earlier,
# or take a new similar photo and save it somewhere accessible.

fabric_test_path = r"D:\Project\Intership Projects\Zynvex Internship\fake-currency-detection\data\raw\Not A Note\4.jpg"

test_img = Image.open(fabric_test_path).convert("RGB").resize((224, 224))
test_array = np.expand_dims(np.array(test_img) / 255.0, axis=0)
pred = three_class_model.predict(test_array, verbose=0)[0]

for name, prob in zip(class_names, pred):
    print(f"{name}: {prob*100:.1f}%")

Fake Notes: 2.7%
Not A Note: 96.5%
Real Notes: 0.8%


## Addressing the Out-of-Distribution Problem

During live testing, the original 2-class model (Fake/Real) classified an image
containing only fabric/background (no currency note) as "Real Note" with 100%
confidence. This revealed a fundamental limitation: the model had no ability to
recognize when an input wasn't a currency note at all, since it was forced to
choose between only two known classes.

**Fix:** A third class, "Not A Note," was introduced using ~165 non-currency
images (Fashion-MNIST samples + supplementary images), and the model architecture
was updated from binary (sigmoid) to 3-class (softmax) classification.

**Result:** The retrained 3-class model correctly identifies the exact fabric
image that previously caused a false positive, now classifying it as "Not A Note"
with 96.5% confidence — validating that the fix directly resolves the discovered
limitation.

| Test Case | 2-Class Model | 3-Class Model |
|---|---|---|
| Fabric background image | "Real Note" (100%) ❌ | "Not A Note" (96.5%) ✅ |